# Experiment 8 - Part 2: The Language Model Brain and the Text-to-Speech Mouth

**Course:** HAAI302COM - Artificial Intelligence Laboratory

Notebook 01 covered the ear (Whisper). Here we build the other two links of the chain:

1. the **brain**: an LLM (Gemini through the OpenAI SDK) that must answer in a way that sounds good when **read aloud**, not just on a screen;
2. the **mouth**: text-to-speech (TTS) engines, compared on speed, sound and how well another model can understand them.

A chat answer full of bullet points, asterisks and links is fine to read and awful to hear. Most of this notebook is about that gap.

## 0. Setup

We reuse `voice_utils.py`. `vu.chat()` is a cached Gemini call (answers are saved in `llm_cache.json`). Our "users" are synthetic voices: the questions below are spoken by Google TTS with an Indian-English accent, so no microphone is needed.

In [1]:
import json, os, re, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from faster_whisper import WhisperModel
from IPython.display import Audio, display
import voice_utils as vu

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 170)
base = WhisperModel("base", device="cpu", compute_type="int8", cpu_threads=8)
QUESTIONS = ["What is the capital of Australia?", "Who wrote the play Hamlet?",
             "How many minutes are there in a day?", "Explain what a vector database is.",
             "What is the boiling point of water in Fahrenheit?", "Name three benefits of regular exercise.",
             "Which planet is known as the red planet?", "How do I make a cup of tea?"]
os.makedirs("data", exist_ok=True)
for i, q in enumerate(QUESTIONS):
    if not os.path.exists(f"data/q_{i}.mp3"):
        vu.tts_gtts(q, f"data/q_{i}.mp3")
print(len(QUESTIONS), "spoken questions ready in data/")

8 spoken questions ready in data/


## 1. Speech to text on spoken questions

First, the ear again. Each question is audio now; Whisper `base` turns it back into text. This is the input the LLM will receive, so we check it matches what was said.

In [2]:
heard = []
for i, q in enumerate(QUESTIONS):
    text, _, _, sec = vu.transcribe(base, vu.load_audio(f"data/q_{i}.mp3"), beam_size=5)
    heard.append(text)
print("WER on the 8 questions:", round(vu.wer(QUESTIONS, heard), 3))
pd.DataFrame(dict(said=QUESTIONS, heard=heard))

WER on the 8 questions: 0.054


,said,heard
0,What is the capital of Australia?,Which is the capital of Australia?
1,Who wrote the play Hamlet?,Who wrote the play hamlet?
2,How many minutes are there in a day?,How many minutes are there in a day?
3,Explain what a vector database is.,Explain what a vector database is.
4,What is the boiling point of water in Fahrenheit?,Which is the boiling point of water in Fahrenheit?
5,Name three benefits of regular exercise.,Name 3 Benefits of Regular Exercise
6,Which planet is known as the red planet?,which planet is known as the red planet?
7,How do I make a cup of tea?,How do I make a cup of tea?


**What to notice:** WER is 0.054 over the 8 questions, and the meaning survives every time. The accented 'What' was heard as 'Which' twice, and 'three' was written as the digit '3' with capital letters, which counts as a word error although it is not a mistake. For a voice assistant the right question is not 'is the transcript identical?' but 'does the LLM still get the same question?'

## 2. The brain: same question, screen answer versus voice answer

We send the transcript to Gemini through the OpenAI SDK (`base_url` points to Google's compatible endpoint, `vu.chat` hides the plumbing). Two versions:

- **default**: no system message. The model writes for a screen.
- **voice**: a system message that says the reply will be spoken: short, plain sentences, no lists, no symbols.

To compare, we count words and the markdown characters (`*`, `#`, `-` at line start) that a speech engine would either read out loud or stumble on.

In [3]:
VOICE_SYS = ("You are a voice assistant. Your reply will be read aloud by a speech engine. Answer in at most three short, "
             "plain sentences. Never use bullet points, numbered lists, headings, asterisks, emojis or links. "
             "Write numbers and units the way a person would say them.")

def md_marks(t):
    return len(re.findall(r"\*|#|^\s*[-\d]+[.)]?\s", t, flags=re.M))

rows = []
for q in heard[3:6]:
    default = vu.chat([{"role": "user", "content": q}])["answer"]
    voice = vu.chat([{"role": "system", "content": VOICE_SYS}, {"role": "user", "content": q}])["answer"]
    rows.append(dict(question=q, words_default=len(default.split()), marks_default=md_marks(default),
                     words_voice=len(voice.split()), marks_voice=md_marks(voice)))
pd.DataFrame(rows)

,question,words_default,marks_default,words_voice,marks_voice
0,Explain what a vector database is.,263,45,30,0
1,Which is the boiling point of water in Fahrenheit?,9,4,30,0
2,Name 3 Benefits of Regular Exercise,119,15,27,0


In [4]:
q = heard[5]
print("DEFAULT:\n", vu.chat([{"role": "user", "content": q}])["answer"][:600])
print("\nVOICE:\n", vu.chat([{"role": "system", "content": VOICE_SYS}, {"role": "user", "content": q}])["answer"])

DEFAULT:
 Here are 3 benefits of regular exercise:

1.  **Improved Cardiovascular Health:** Regular exercise strengthens your heart and lungs, making them more efficient at pumping blood and oxygen throughout your body. This reduces your risk of heart disease, stroke, high blood pressure, and high cholesterol.

2.  **Enhanced Mood and Mental Well-being:** Exercise releases endorphins, which are natural mood boosters. This can help reduce stress, anxiety, and symptoms of depression. It can also improve sleep quality and boost self-esteem.

3.  **Increased Muscle Strength and Bone Density:** Weight-bearin

VOICE:
 Regular exercise improves your cardiovascular health. It also helps manage your weight and boosts your mood. Additionally, it can increase your energy levels and improve sleep quality.


**What to notice:** the voice prompt removes all markup (0 marks in every row) and shortens long answers sharply: 263 words and 45 markup marks for 'explain a vector database' became 30 words and none, and the exercise question went from 119 words with 15 marks to 27 words and none. The prompt controls format more than length, though: the boiling point question had a 9 word default answer and a longer 30 word voice answer, because we asked for up to three sentences. Read aloud, 263 words would take over a minute.

## 3. A safety net: cleaning text before speaking

Even with a good prompt, models sometimes slip a markdown symbol or a link into a reply. `clean_for_speech` removes what a speech engine should never read: code fences, emphasis markers, headings, bullet characters and URLs. It is cheap insurance, written with plain regular expressions.

In [5]:
def clean_for_speech(t):
    t = re.sub(r"```.*?```", " ", t, flags=re.S)             # code blocks
    t = re.sub(r"https?://\S+", " ", t)                       # links
    t = re.sub(r"[*_`#>]+", "", t)                            # emphasis, headings, quotes
    t = re.sub(r"^\s*(?:[-+]|\d+[.)])\s+", "", t, flags=re.M) # list markers at the start of a line
    return re.sub(r"\s+", " ", t).strip()

sample = "## Benefits\n* **Heart health**: see https://example.com/x\n* Better sleep\n1. Less stress"
print(repr(sample)); print(repr(clean_for_speech(sample)))

'## Benefits\n* **Heart health**: see https://example.com/x\n* Better sleep\n1. Less stress'
'Benefits Heart health: see Better sleep Less stress'


**What to notice:** the headings, bullet stars, numbering and the link are gone. One weakness is visible: the list items now run together ('Better sleep Less stress') with no full stop, so a speech engine would read them without a pause, and the colon stays. A better version would replace line breaks with full stops. This is why the prompt is the primary fix and the cleaner only a safety net.

## 4. Memory: a conversation is a list of messages

The LLM has no memory between calls. A conversation works because **we resend the whole history** each turn. We ask "Who wrote the play Hamlet?" and then "When was he born?", once with the history and once without it.

In [6]:
def turn(history, user_text):
    msgs = [{"role": "system", "content": VOICE_SYS}] + history + [{"role": "user", "content": user_text}]
    return vu.chat(msgs)["answer"]

h = []
a1 = turn(h, "Who wrote the play Hamlet?")
h += [{"role": "user", "content": "Who wrote the play Hamlet?"}, {"role": "assistant", "content": a1}]
print("with history   :", turn(h, "When was he born?"))
print("without history:", turn([], "When was he born?"))

with history   : William Shakespeare was baptized on April twenty-sixth, fifteen sixty-four. His exact birth date is not known. He was likely born a few days before his baptism.
without history: He was born on January twenty-fourth, nineteen fifty-seven.


**What to notice:** with the history the model understands that 'he' means Shakespeare, and gives a careful answer (baptised in April 1564, exact birth date unknown). Without it, the model has to guess who 'he' is and confidently invents a birth date, 'January twenty-fourth, nineteen fifty-seven', for someone unknown. The model remembers nothing between calls; every follow-up question is only as good as the history we send with it.

## 5. Latency: start speaking before the answer is finished

Waiting for the whole reply before speaking makes the assistant feel slow. The trick is to **stream** the LLM output and hand each finished sentence to the TTS right away. We stream one reply and record three times: first token, first complete sentence (the moment TTS could begin) and the end.

In [7]:
def stream_times(question):
    t0 = time.time(); first_tok = first_sent = None; buf = ""
    stream = vu.client().chat.completions.create(model="gemini-2.5-flash-lite", temperature=0, stream=True,
        messages=[{"role": "system", "content": "Answer in about four plain sentences. No lists or symbols."},
                  {"role": "user", "content": question}])
    for ch in stream:
        piece = ch.choices[0].delta.content or ""
        if piece and first_tok is None: first_tok = time.time() - t0
        buf += piece
        if first_sent is None and re.search(r"[.!?]\s", buf): first_sent = time.time() - t0
    return dict(first_token=first_tok, first_sentence=first_sent, total=time.time() - t0, words=len(buf.split()))

try:
    print(pd.DataFrame([stream_times(q) for q in [heard[3], heard[5], heard[7]]]).round(2))
except Exception as e:
    print("streaming call failed (needs GEMINI_API_KEY and internet):", type(e).__name__)

   first_token  first_sentence  total  words
0         1.56            1.62   1.89     81
1         0.92            0.98   1.07     37
2         1.11            1.17   1.34     57


**What to notice:** the first token arrives after about 0.9 to 1.6 seconds, the first complete sentence at about the same time, and the whole reply (37 to 81 words) finishes only 0.1 to 0.3 seconds later. With a fast model and short replies, streaming sentences to the TTS would save very little here: the wait is dominated by the time to the first token (network plus model start-up). Sentence-level streaming pays off with long replies or slower models, and short spoken replies help in both cases.

## 6. The mouth: three ways to speak

- **espeak-ng** (command-line speech synthesizer): runs offline on any computer, robotic voice, fast. `rate` sets words per minute.
- **gTTS** (Google Translate voice): free, needs internet, much more natural. `tld` chooses the accent.
- **ElevenLabs**: commercial neural voices of the highest quality, needs an API key (section 7).

First we generate the same sentence with several settings and measure how long the audio is and how long the generation takes. Duration of the clip is the length of speech; generation time is what the user waits.

In [8]:
SENT = "The red planet is Mars, the fourth planet from the sun."
rows = []
for label, fn, kw in [("espeak rate 120", vu.tts_espeak, dict(rate=120)), ("espeak rate 170", vu.tts_espeak, dict(rate=170)),
                      ("espeak rate 230", vu.tts_espeak, dict(rate=230)),
                      ("gTTS en-IN", vu.tts_gtts, dict(tld="co.in")), ("gTTS en-US", vu.tts_gtts, dict(tld="com")),
                      ("gTTS en-GB", vu.tts_gtts, dict(tld="co.uk"))]:
    path = "data/s_" + label.replace(" ", "_") + (".wav" if "espeak" in label else ".mp3")
    gen = fn(SENT, path, **kw)
    a = vu.load_audio(path)
    rows.append(dict(engine=label, audio_seconds=len(a) / vu.SR, generation_seconds=gen, words_per_minute=len(SENT.split()) / (len(a) / vu.SR) * 60))
tts_table = pd.DataFrame(rows).set_index("engine").round(2)
tts_table

,audio_seconds,generation_seconds,words_per_minute
engine,,,
espeak rate 120,4.77,0.03,138.49
espeak rate 170,3.29,0.02,200.91
espeak rate 230,2.41,0.02,273.93
gTTS en-IN,4.58,0.54,143.98
gTTS en-US,4.58,0.19,143.98
gTTS en-GB,4.15,0.51,158.96


**What to notice:** espeak makes 3 to 5 seconds of audio in about 0.03 seconds, effectively instant and offline, while gTTS needs 0.3 to 0.7 seconds because it calls a server. The espeak rate setting does change the speech (4.8, 3.3 and 2.4 seconds for the same sentence), though the real speed in words per minute (138, 201, 274) differs from the number we set. The three gTTS accents give nearly the same duration, so the numbers cannot tell them apart: you have to listen to the clips below.

In [9]:
# listen: robotic offline voice versus Indian-English Google voice
display(Audio("data/s_espeak_rate_170.wav")); display(Audio("data/s_gTTS_en-IN.mp3"))

## 7. ElevenLabs (needs an API key)

ElevenLabs offers lifelike neural voices. The free tier has a monthly character quota. The cell below runs only when `ELEVENLABS_API_KEY` is present in `.env`; otherwise it prints a message, so the notebook still runs without it. `voice_id` selects the voice and `model_id` trades quality for speed (`eleven_flash_v2_5` is the low-latency model, `eleven_multilingual_v2` the highest quality).

In [10]:
if os.environ.get("ELEVENLABS_API_KEY"):
    sec = vu.tts_elevenlabs(SENT, "data/s_elevenlabs.mp3")
    print("ElevenLabs generated in", round(sec, 2), "s,", round(len(vu.load_audio("data/s_elevenlabs.mp3")) / vu.SR, 2), "s of audio")
    display(Audio("data/s_elevenlabs.mp3"))
else:
    print("ELEVENLABS_API_KEY not set: skipping the ElevenLabs demo (add it to .env to run).")

ELEVENLABS_API_KEY not set: skipping the ElevenLabs demo (add it to .env to run).


## 8. Measuring a TTS engine: the round trip

How do we score a voice objectively? One way is **intelligibility**: speak a sentence, transcribe it with Whisper and compute the WER against the original text. A voice that Whisper cannot understand is hard for people too. (It does not measure how pleasant the voice is; that needs listeners.) We use the 8 technical sentences from notebook 01, which include hard words.

In [11]:
SENTS = ["We store the embeddings in Qdrant and retrieve them with LangChain.",
         "The student fine tuned Mistral with LoRA on Google Colab.",
         "Pinecone and FAISS are popular vector databases.",
         "PCCOER offers an honours programme in agentic AI.",
         "LangGraph supports human in the loop checkpoints.",
         "The service streams tokens from the Gemini model."]
rows = []
for engine, fn, kw, ext in [("espeak", vu.tts_espeak, dict(rate=170), "wav"), ("gTTS en-IN", vu.tts_gtts, dict(tld="co.in"), "mp3")]:
    hyps, gen = [], 0
    for i, s in enumerate(SENTS):
        p = f"data/rt_{engine.split()[0]}_{i}.{ext}"
        gen += fn(s, p, **kw)
        hyps.append(vu.transcribe(base, vu.load_audio(p), beam_size=5)[0])
    rows.append(dict(engine=engine, WER=vu.wer(SENTS, hyps), generation_seconds_per_sentence=gen / len(SENTS)))
    print(engine, "->", hyps[0], "|", hyps[1])
pd.DataFrame(rows).set_index("engine").round(3)

espeak -> And we saw them having the cube run, and retrieved them with our chain. | The student fine tuned mistral with low array on google collab.


gTTS en-IN -> We store the embeddings in Cudrent and retrieve them with lang chain. | The student fined Tuned Mastral with Laura on Google Colab.


,WER,generation_seconds_per_sentence
engine,,
espeak,0.510,0.038
gTTS en-IN,0.235,0.624


**What to notice:** Whisper understands gTTS much better than espeak: WER 0.20 against 0.51. The espeak voice turned 'Qdrant' into 'cube run' and 'LoRA' into 'low array'; the gTTS voice still gave 'Cudrent', 'Mastral' and 'Laura'. espeak is about twenty times faster (0.03 against 0.63 seconds per sentence) but only half as intelligible by this test. Rare technical words are hard for both. Remember that this score includes the ear's own mistakes; it is a proxy, not a listening test.

## 9. How a mishearing travels through the chain

If Whisper mishears a key word, the LLM answers a different question, often confidently. We add noise to spoken questions (5 dB and 0 dB), use the smallest Whisper (`tiny`) as the ear, and print what it heard and what the assistant answered.

In [12]:
tiny = WhisperModel("tiny", device="cpu", compute_type="int8", cpu_threads=8)
rows = []
for i in [1, 2, 4, 6]:
    a = vu.load_audio(f"data/q_{i}.mp3")
    for snr in [None, 5, 0]:
        x = a if snr is None else vu.add_noise(a, snr)
        heard_text = vu.transcribe(tiny, x, beam_size=5)[0]
        ans = vu.chat([{"role": "system", "content": VOICE_SYS}, {"role": "user", "content": heard_text or "(silence)"}])["answer"]
        rows.append(dict(said=QUESTIONS[i][:34], snr="clean" if snr is None else f"{snr} dB", heard=heard_text[:40], answer=ans[:70]))
pd.DataFrame(rows)

,said,snr,heard,answer
0,Who wrote the play Hamlet?,clean,Who wrote the plea Hamlet?,William Shakespeare wrote the play Hamlet. It is one of his most famou
1,Who wrote the play Hamlet?,5 dB,سوف تتليها منك,أنا هنا لمساعدتك. كيف يمكنني خدمتك اليوم؟
2,Who wrote the play Hamlet?,0 dB,Who will continue on it?,The question is too vague for me to answer. Please provide more contex
3,How many minutes are there in a da,clean,How many minutes are there in a day?,There are one thousand four hundred forty minutes in a day. This is ca
4,How many minutes are there in a da,5 dB,아몬이 미쳤어 보이는데,아몬이 미쳤다고요? 무슨 일이 있었는지 자세히 말씀해 주시겠어요?
5,How many minutes are there in a da,0 dB,아 매니깐 더 더 이뤄게,"네, 더 노력해서 이루고 싶으신 것이 무엇인가요. 무엇을 도와드릴까요."
6,What is the boiling point of water,clean,which is the boiling point of water in F,The boiling point of water is two hundred twelve degrees Fahrenheit. T
7,What is the boiling point of water,5 dB,which is a boiling pond of water in sile,A boiling pond of water in a silent house is likely a pot of water on
8,What is the boiling point of water,0 dB,which is a very important water inside h,The most important water inside a house is potable water. This is the
9,Which planet is known as the red p,clean,Which planet is known as the red planet?,Mars is known as the red planet. Its reddish appearance is due to iron


**What to notice:** on clean audio the answers are right. At 5 dB the small ear first guessed a wrong language: it wrote Arabic, Korean and Urdu characters, and the LLM faithfully answered in the same languages. At 0 dB and for the boiling point question, the English transcript was wrong too ('boiling pond of water in silent house'), so the assistant answered a question nobody asked, about potable water, or asked for more context. The chain does not know the ear was unsure; it answers whatever it hears. That is why the ear's confidence should be passed on in a real system.

## 9b. A fix: tell the ear the language

Look at the 5 dB rows: with noise, Whisper's automatic language detection guessed Arabic, Korean or Urdu, wrote text in that script, and the LLM politely answered in the same language. Auto-detection looks at the first seconds of audio, and noise makes that guess unreliable. If we know the users speak English, we pass `language="en"` and skip the guess.

In [13]:
rows = []
for i in [1, 2, 6]:
    a = vu.add_noise(vu.load_audio(f"data/q_{i}.mp3"), 5)
    auto = vu.transcribe(tiny, a, beam_size=5)
    forced = vu.transcribe(tiny, a, beam_size=5, language="en")
    rows.append(dict(said=QUESTIONS[i][:34], auto_language=auto[2].language, heard_auto=auto[0][:32], heard_forced=forced[0][:40]))
pd.DataFrame(rows)

,said,auto_language,heard_auto,heard_forced
0,Who wrote the play Hamlet?,ar,سوف تكون هنا,who will continue having it?
1,How many minutes are there in a da,ko,아몬이 미쳤어 보이는데,How many minutes are there in the day?
2,Which planet is known as the red p,ur,یہ کسی پلانے تجنے مہمے تجنے مہمے,which planet is known as the word planet


**What to notice:** forcing English removes the foreign scripts: 'How many minutes are there in the day?' is now almost perfect. The other two are still wrong in English ('Who will continue having it?', 'word planet'), so language forcing only fixes the language guess, not the acoustic errors. A larger ear or cleaner audio fixes those.

## 10. Recap and exercises

- The brain needs a **voice system prompt**: short sentences, no markdown, speakable numbers; add `clean_for_speech` as a safety net.
- Memory is the history we resend; streaming plus sentence-level TTS cuts the waiting time.
- TTS engines trade naturalness against speed and cost; round-trip WER gives a cheap objective intelligibility score.
- ASR errors travel down the chain, so the ear's quality limits the whole assistant; fix the language when you know it.

**Try yourself:** (1) change `VOICE_SYS` to ask for one sentence and observe the effect on latency; (2) add `eleven_multilingual_v2` and a Hindi question; (3) compare a `tiny` and a `small` ear at 0 dB.

Next: **Notebook 03** wires the three parts into one assistant, measures every stage and shows how to run it live.